In [1]:
from datetime import datetime
import pandas as pd
import numpy as np
from tqdm import tqdm


path_ent = '/home/tbadie/Bureau/data/data_entrepot_outils/'
path_ext = '/home/tbadie/Bureau/data/temp/'
donnees = {}

def import_donnees(donnees_name, path_data, sep, index_col=None):
    donnees[donnees_name] = pd.read_csv(path_data+donnees_name+'.csv', sep = sep, index_col=index_col, low_memory=False).replace({'\r\n': '\n'}, regex=True)

def import_all_donnees(donnees_names, path_data, sep = ',', index_col=None, verbose=False):
    for donnees_name in tqdm(donnees_names) : 
        if(verbose) :
            print(" - ", donnees_name)
        import_donnees(donnees_name, path_data, sep, index_col=index_col)

tables = [
        'synthetise',
        'sdc',
        'typologie_assol_can_realise',
        'typologie_can_rotation_synthetise',
        'entite_unique_par_sdc_nettoyage',
        'sdc_realise_performance',
        'synthetise_synthetise_performance',
        'intervention_synthetise_agrege',
        'intervention_realise_agrege',

        'identification_pz0',
        'zone',
        'parcelle'
]

import_all_donnees(tables, path_ent, sep = ',', verbose=False)

100%|██████████| 12/12 [00:35<00:00,  2.99s/it]


In [2]:
def filtered_entities_sdc_level(donnees):
    """
    Cette fonction permet de filtrer les entités que l'on veut présente dans le dataframe principale de la table SDC du magasin DIRODUR.
    Globalement il y a en premier lieu un filtre sur les entités avec intervention (par construction),
      puis des filtre sur la filiere GCPE ; 
      sur une entité unique par sdc ; 
      sur les années trop veilles (ne devrait pas être présente) ou trop récentes (pas encore assez consolidées) ; 
      et enfin sur les alertes (petite expection si les cultures sont des prairies temporaires plus de 50% du temps).
    """

    
    lst_alerte_col = ['alerte_ferti_n_tot', 'alerte_ift_cible_mil_chim_tot_hts',
        'alerte_ift_cible_mil_f', 'alerte_ift_cible_mil_h',
        'alerte_ift_cible_mil_i', 'alerte_ift_cible_mil_biocontrole',
        'alerte_co_irrigation_std_mil', 'alerte_msn_std_mil_avec_autoconso',
        'alerte_nombre_interventions_phyto', 'alerte_pb_std_mil_avec_autoconso',
        'alerte_rendement', 'alertes_charges', 'alerte_cm_std_mil',
        'alerte_co_semis_std_mil', 'alerte_tps_travail_total']

    int_r = donnees['intervention_realise_agrege'][['id','sdc_id']]
    sdc_real = donnees['sdc_realise_performance'][['sdc_id']+lst_alerte_col]
    sdc = donnees['sdc'][['id','filiere','campagne']].rename(columns={'id':'sdc_id'})
    typo_re = donnees['typologie_assol_can_realise'][['sdc_id','typocan_assol_dvlp']]

    int_s = donnees['intervention_synthetise_agrege'][['id','synthetise_id','sdc_id']]
    sdc_synth = donnees['synthetise_synthetise_performance'][['synthetise_id']+lst_alerte_col]
    typo_sy = donnees['typologie_can_rotation_synthetise'][['synthetise_id','typocan_rotation']]

    synth = donnees['synthetise'][['id']].rename(columns={'id':'synthetise_id'}) # besoin que pour la fin
    unique_entity = donnees['entite_unique_par_sdc_nettoyage']

    # 0. Avant tout : Par le jeu des merge on ne garde que les synthé ou sdc qui ont des interventions !
    sdc_real = int_r.merge(sdc_real, on='sdc_id', how='left').merge(sdc, on='sdc_id', how='left').merge(typo_re, on='sdc_id', how='left')
    sdc_real = sdc_real.groupby('sdc_id').first().reset_index().drop(columns='id')

    sdc_synth = int_s.merge(sdc_synth, on='synthetise_id', how='left').merge(sdc, on='sdc_id', how='left').merge(typo_sy, on='synthetise_id', how='left')
    sdc_synth = sdc_synth.groupby('synthetise_id').first().reset_index().drop(columns='id')

    # 1. on filtre selon la filière GCPE
    sdc_real = sdc_real.loc[sdc_real['filiere'].isin(['POLYCULTURE_ELEVAGE','GRANDES_CULTURES'])]
    sdc_synth = sdc_synth.loc[sdc_synth['filiere'].isin(['POLYCULTURE_ELEVAGE','GRANDES_CULTURES'])]

    # 2. on filtre grace à l'outil d'entité unique par sdc
    # il regarde dans un sdc s'il y a plusieurs synthétisé et ou zone de réalisé et en choisi un seul
    lst_unqiue_entity_real = unique_entity.loc[unique_entity['entite_retenue']=='realise_retenu','sdc_id'].tolist()
    sdc_real = sdc_real.loc[sdc_real['sdc_id'].isin(lst_unqiue_entity_real)]

    lst_unqiue_entity_synth = unique_entity.loc[unique_entity['entite_retenue']!='realise_retenu','entite_retenue'].to_list()
    sdc_synth = sdc_synth.loc[sdc_synth['synthetise_id'].isin(lst_unqiue_entity_synth)]

    # 3. on va filtré l'année en cours pour le magasin car les données sont surement en cours de saisie ou en cours de consolidation par les IR. En gros au 1° avril on accorde l'ajout de l'année n-1
    # Définir les années limites (seuils strictes !)
    if datetime.now().month <= 3 :  annees_max = datetime.now().year - 1
    else :                          annees_max = datetime.now().year
    annees_trop_vieille = 2004 # des pz0 attendues jusqu'en 2005

    sdc_synth = sdc_synth.loc[((pd.to_numeric(sdc_synth['campagne'], errors='coerce')) > annees_trop_vieille) & 
                            ((pd.to_numeric(sdc_synth['campagne'], errors='coerce')) < annees_max)]
    sdc_real = sdc_real.loc[((pd.to_numeric(sdc_real['campagne'], errors='coerce')) > annees_trop_vieille) & 
                            ((pd.to_numeric(sdc_real['campagne'], errors='coerce')) < annees_max)]


    # 4. on filtre par alertes
    list_alerte_ok = [
        "Pas d'alerte", 
        "Cette alerte n'existe pas dans cette filière", 
        "Cette alerte n'existe pas encore dans cette filière"
    ]

    def filtrer_alertes(df, lst_alerte_col, list_alerte_ok, name_culture_col):
        # Masque pour les colonnes d'alertes sauf 1
        autres_colonnes = [col for col in lst_alerte_col if col != 'alerte_cm_std_mil']
        mask_autres = df[autres_colonnes].apply(
            lambda x: x.isin(list_alerte_ok) | x.isna()
        ).all(axis=1)

        # Mask pour les alertes de CM lorsque la culture est majoritairement de la prairie
        mask_cm = (
            df['alerte_cm_std_mil'].isin(list_alerte_ok) |
            df['alerte_cm_std_mil'].isna() |
            ((df['alerte_cm_std_mil'].str.contains('<', na=False)) & (df[name_culture_col] == "prairie temporaire >= 50 % assolement"))
            )

        mask_final = mask_autres & mask_cm
        return df[mask_final]


    sdc_real = filtrer_alertes(sdc_real, lst_alerte_col, list_alerte_ok, 'typocan_assol_dvlp')
    sdc_synth = filtrer_alertes(sdc_synth, lst_alerte_col, list_alerte_ok, 'typocan_rotation')

    # On prend tout les sdc, et tout les synthetise, on leur tag s'ils appartiennent ou non à dirodur
    final_real = sdc[['sdc_id']]
    final_real['in_dirodur'] = np.where(final_real['sdc_id'].isin(sdc_real['sdc_id']), True, False)

    final_synth = synth[['synthetise_id']]
    final_synth['in_dirodur'] = np.where(final_synth['synthetise_id'].isin(sdc_synth['synthetise_id']), True, False)

    # On exporte la liste pour les realise et la liste pour les synthe
    return final_real, final_synth

In [4]:
# On importe les données
sdc = donnees['sdc'][['id','code','campagne','modalite_suivi_dephy','code_dephy','type_agriculture']].rename(columns={'id':'sdc_id','code':'sdc_code'})
synthetise = donnees['synthetise'][['id','campagnes','sdc_id']].rename(columns={'id':'synthetise_id'})
zone = donnees['zone'][['id','parcelle_id']].rename(columns={'id':'entite_id'})
parcelle = donnees['parcelle'][['id','sdc_id']].rename(columns={'id':'parcelle_id'})
outil_pz0 = donnees['identification_pz0']

# On importe la fonction de filtration des dataframe SDC en réal et en synth
sdc_realise_filt, synthetises_filt = filtered_entities_sdc_level(donnees)

# On crée les df, et on les filtre pour qu'ils soient dans dirodur
df_R = sdc_realise_filt.loc[sdc_realise_filt['in_dirodur']]\
.merge(sdc, on='sdc_id', how='left')\
    .drop(columns=['in_dirodur'])

df_S = synthetises_filt.loc[synthetises_filt['in_dirodur']]\
    .merge(synthetise, on='synthetise_id', how='left')\
        .merge(outil_pz0.rename(columns={'entite_id':'synthetise_id'}), on='synthetise_id', how='left')\
            .merge(sdc, on='sdc_id', how='left')\
                .drop(columns=['in_dirodur'])

# L'outil d'identification des pz0 a la particularité d'etre au niveau de la zone, on fait en sorte d'avoir les infos niveau SDC
def list_to_scalar(serie):
    unique_values = list(serie.dropna().unique())
    if len(unique_values) == 0:
        return None
    if len(unique_values) == 1:
        return unique_values[0]
    return unique_values
    
zones_w_pz0 = zone.merge(parcelle, on='parcelle_id', how='left').merge(outil_pz0, on='entite_id', how='left')
zones_w_pz0 = zones_w_pz0.groupby('sdc_id')['pz0'].apply(list_to_scalar, include_groups=False).reset_index()

if len(zones_w_pz0.loc[zones_w_pz0['pz0'].apply(lambda x: isinstance(x, list))] ) > 0 :
    raise ValueError("Il y a des sdc réalisé avec plusieurs identification différentes selon leur zones")

df_R = df_R.merge(zones_w_pz0, on='sdc_id', how='left')

# On crée le df principal en concaténant réalisé et synthétisé
df_R['pz0'] = np.where(df_R['modalite_suivi_dephy']=='DETAILLE',
                    df_R['pz0'], 
                    np.where(df_R['modalite_suivi_dephy'].isna(),'non_DEPHY', 'non_suivi'))
df_S['pz0'] = np.where(df_S['modalite_suivi_dephy']=='DETAILLE',
                    df_S['pz0'], 
                    np.where(df_S['modalite_suivi_dephy'].isna(),'non_DEPHY', 'non_suivi'))
df = pd.concat([
    df_S[['sdc_id', 'sdc_code', 'code_dephy', 'type_agriculture', 'campagne', 'pz0', 'synthetise_id', 'campagnes']],
    df_R[['sdc_id', 'sdc_code', 'code_dephy', 'type_agriculture', 'campagne', 'pz0']]
    ])

df.pz0 = df.pz0.fillna('non_DEPHY') # ceux dont la modalité de suivi est NA

# On modifie les modalités incorrectes de pz0 pour qu'elles soient regroupées sous la modalité 'post'. Seulement pour celles qui ne détecte pas de pz0 fiable.
df['pz0'] = np.where(df['pz0'].isin([
    'incorrect : saisie pz0 non acceptable',
    'incorrect : aucun pz0 saisi',
    'incorrect : chevauchement pz0',
    'incorrect : saisie de plusieurs pz0']), 'post', df['pz0'])
# Les modalités incorrects de camapgne non attendue ou de code dephy inconnue sont enlevé car même les points_B peuvent être faux.
df = df.loc[~df['pz0'].isin([
    "incorrect : campagne non-attendue",
    "incorrect : code dephy inconnu"
])]


/tmp/ipykernel_292965/752856590.py:93: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_real['in_dirodur'] = np.where(final_real['sdc_id'].isin(sdc_real['sdc_id']), True, False)


In [5]:
bckup = df.copy()

In [13]:

# On crée les fonction permettant l'identification des etats temporels
def extract_years(row):
    """ Extrait les années sous forme de liste d'int de la colonne campagnes pour les synthétisés et de la colonne campagne pour les réalisés """
    years = set()
    if pd.notna(row['campagne']):
        years.add(int(row['campagne']))
    if pd.notna(row['campagnes']):
        years.update(int(y) for y in row['campagnes'].split(', '))
    return sorted(years)

def label_pz0_status(df):
    """ on modifie un peu les label de l'outil d'identification des pz0 pour crée le début de 'état_temporel'. 
    Typiquement on check s'il y a bien au moins 2 pz0 tagué pour un numéro DEPHY, si ce n'est pas le cas on regarde si ils ont été filtré par la fonction util ou si l'outil était déjà sans pz0 pour ce code DEPHY. """
    df_pz0 = df[df['pz0'] == 'pz0'].copy()
    df_pz0['all_years'] = df_pz0.apply(extract_years, axis=1)
    grouped = df_pz0.groupby('code_dephy')['all_years'].agg(lambda x: set().union(*x))
    valid_groups = grouped[grouped.apply(len) >= 2].index.tolist()
    not_incorrect_cd = df.loc[df['pz0'].isin(['pz0','post']),'code_dephy'].tolist()

    df['serie_tempo'] = df.apply(
        lambda row:
            'sans_pz0' if row['code_dephy'] not in not_incorrect_cd and row['code_dephy'] not in valid_groups
            else 'pz0_filtres' if row['code_dephy'] in not_incorrect_cd and row['code_dephy'] not in valid_groups
            else 'pz0' if row['pz0'] == 'pz0' and row['code_dephy'] in valid_groups
            else row['pz0'],
        axis=1
    )

    return df

def find_last_n_year(years, n):
    """ recheche la dernière séquence des n années les plus récentes et consécutives"""
    for i in range(len(years) - n, -1, -1):
        window = years[i:i+n]
        if all(window[j+1] - window[j] == 1 for j in range(len(window)-1)):
            return window
    return None
    
def find_last_consecutive_year_sequence(years):
    """ 
    Utilise find_last_n_year() pour repéré les séquences les plus récentes de n années consécutives. 
    Puis fait le choix entre la séquence de 3 années et de 2 année. On privilégie les séquences les plus récentes, 
    puis les séquences les plus grande (3 > 2) 
    """

    if not years:
        return []

    last_3 = find_last_n_year(years, 3) if len(years) >= 3 else None
    last_2 = find_last_n_year(years, 2) if len(years) >= 2 else None

    if last_3 and last_2:
        return last_3 if last_3[-1] >= last_2[-1] else last_2
    return last_3 or last_2 or []

def update_final_status_for_code_dephy_without_point_B(df, codes_with_consecutive):
    """ 
    Dernière fonction a être appelé. 
    Permet de check s'il y a des points_B parmi chaque code DEPHY. 
    Si ce n'est pas le cas, ajoute un message d'erreur qui correspond au cas. 
    """

    for code in list(df['code_dephy'].unique()):
        if code not in codes_with_consecutive:
            mask = df['code_dephy'] == code
            if (df.loc[mask, 'serie_tempo'] == 'sans_pz0').any():
                df.loc[mask, 'serie_tempo'] = 'ni_pz0_ni_point_B'
            elif (df.loc[mask, 'serie_tempo'] == 'pz0_filtres').any():
                df.loc[mask, 'serie_tempo'] = 'pz0_filtres_et_sans_point_B'
            else:
                df.loc[mask, 'serie_tempo'] = 'sans_point_B'

    return df

def get_last_consecutive_years(df):
    """ 
    fonction principale qui va extraire pour chaque code DEPHY la séquence des dernières années consécutive parmis un df sans les pz0. 
    Puis va checker dans le df (tout compris cette fois) chaque sdc : s'il est un pz0 ou qu'il fait parti des code dephy sans pz0, 
    on ne modifie pas la ligne et on garde en mémoire que le code DEPHY pourrait contenir des points_B mais n'a pas de pz0 ; 
    s'il est autre chose (post ou incorrect uniquement pour le sdc associé) on va chercher la ou les campagnes du sdc, 
    si au moins une est présente dans la liste des années retenues pour être des point_B on modifie l'état temporel en 'point_B'. 
    Si une ligne a une année supérieur à l'année maximal du point B, on la tague 'point_C' Enfin on utilise la fonction 
    update_final_status_for_code_dephy_without_point_B(). 
    """

    df_non_pz0 = df[df['etat_temporel'] != 'pz0'].copy()
    df_non_pz0['all_years'] = df_non_pz0.apply(extract_years, axis=1)
    grouped = df_non_pz0.groupby('code_dephy')['all_years'].agg(lambda x: sorted(set().union(*x)))
    consecutive_years = grouped.apply(find_last_consecutive_year_sequence)

    df = df.set_index('sdc_id')
    df['all_years'] = df.apply(extract_years, axis=1)
    codes_with_consecutive = set()
    for code, years in consecutive_years.items():
        if years:
            codes_with_consecutive.add(code) # garde en mémoire les code ok pour le update final
            mask = (df['code_dephy'] == code) & \
                    (~df['etat_temporel'].isin(
                        ['pz0',
                        'incorrect : campagne non-attendue',
                        'incorrect : code dephy inconnu', 
                        'non_DEPHY', 
                        'non_suivi']
                        ))
            for idx, row in df[mask].iterrows():
                if any(y in years for y in row['all_years']):
                    df.loc[idx, 'etat_temporel'] = 'point_B'
                elif all(y > max(years) for y in row['all_years']):
                    df.loc[idx, 'etat_temporel'] = 'point_C'
    df = df.reset_index()

    df = update_final_status_for_code_dephy_without_point_B(df, codes_with_consecutive)
    return df

def add_etat_temporel_column(df):
    """ Dernière fonction qui wrap le tout et crée les point_I intermédiaire, et met en forme le df final (drop et sort). """

    # On rajoute une regle avant tout : si le type_agriculture est différent au seins du pz0, on supprime le pz0 !
    # on entend par différent AB != AConv. Information obligatoire et En conversion ne sont pas considéré comme différent de l'un ou de l'autre.
    type_agri_pz0 = (
        df[df["pz0"] == "pz0"].groupby(["code_dephy"])["type_agriculture"]
        .apply(lambda x: {"Agriculture conventionnelle", "Agriculture biologique"}.issubset(set(x.dropna())))
    )
    print(f"Il y a {len(type_agri_pz0.unique())} numéros DEPHY qui ont des entités pz0 ayant des types d'agricultures différentes (AB vs AConv, les autres types n'étant ni considérés comme l'un ni comme l'autre)")
    df = df[
        ~((df["pz0"] == "pz0") & (df["code_dephy"].isin(type_agri_pz0)))
    ]
    

    df['serie_tempo'] = pd.NA
    df['etat_temporel'] = df['pz0']

    df = label_pz0_status(df)
    df = get_last_consecutive_years(df)

    code_pz0 = df.loc[df["etat_temporel"] == "pz0", "code_dephy"]
    code_pb = df.loc[df["etat_temporel"] == "point_B", "code_dephy"]
    codes_complete_serie = df.loc[(df["code_dephy"].isin(code_pz0)) & (df["code_dephy"].isin(code_pb)), "code_dephy"]
    df.loc[(df["code_dephy"].isin(codes_complete_serie)) & (df['etat_temporel'] == 'post'), "etat_temporel"] = "point_I"
    df.loc[(~df["code_dephy"].isin(codes_complete_serie)) & (df['etat_temporel'] == 'post'), "etat_temporel"] = "point_A"


    df['etat_temporel'] = np.where(df['etat_temporel'] == 'post', 'point_I', df['etat_temporel'])
    df['serie_tempo'] = np.where(df['serie_tempo'].isin(['post','pz0']), 'serie_complete', df['serie_tempo'])


    df.drop(columns=['pz0','all_years'], inplace=True)

    return df.sort_values(['code_dephy','campagne'])


In [14]:
df = bckup.copy()
add_etat_temporel_column(df)

Il y a 2 numéros DEPHY qui ont des entités pz0 ayant des types d'agricultures différentes (AB vs AConv, les autres types n'étant ni considérés comme l'un ni comme l'autre)


,sdc_id,sdc_code,code_dephy,type_agriculture,campagne,synthetise_id,campagnes,serie_tempo,etat_temporel
6584,fr.inra.agrosyst.api.entities.GrowingSystem_d2...,2b98900e-b3a4-4701-9e64-467db8426066,\tGCF34377,Agriculture conventionnelle,2018,NaN,NaN,pz0_filtres,point_A
4721,fr.inra.agrosyst.api.entities.GrowingSystem_c7...,2b98900e-b3a4-4701-9e64-467db8426066,\tGCF34377,Agriculture conventionnelle,2019,NaN,NaN,pz0_filtres,point_B
6316,fr.inra.agrosyst.api.entities.GrowingSystem_89...,2b98900e-b3a4-4701-9e64-467db8426066,\tGCF34377,Agriculture conventionnelle,2020,NaN,NaN,pz0_filtres,point_B
7028,fr.inra.agrosyst.api.entities.GrowingSystem_03...,2b98900e-b3a4-4701-9e64-467db8426066,\tGCF34377,Agriculture conventionnelle,2021,NaN,NaN,pz0_filtres,point_B
5560,fr.inra.agrosyst.api.entities.GrowingSystem_a1...,2b98900e-b3a4-4701-9e64-467db8426066,\tGCF34377,Agriculture conventionnelle,2023,NaN,NaN,pz0_filtres,point_C
...,...,...,...,...,...,...,...,...,...
7121,fr.inra.agrosyst.api.entities.GrowingSystem_c8...,46661da9-dd6d-41d9-a98d-cbd302280eca,NaN,Agriculture conventionnelle,2024,NaN,NaN,sans_pz0,non_DEPHY
7166,fr.inra.agrosyst.api.entities.GrowingSystem_89...,2f9581ff-bb87-4a08-b2f0-57347d9b6b51,NaN,Agriculture conventionnelle,2024,NaN,NaN,sans_pz0,non_DEPHY
4967,fr.inra.agrosyst.api.entities.GrowingSystem_99...,db0de021-e5b0-4f51-9d1d-6fa3ceeb6eb3,NaN,Agriculture conventionnelle,2025,NaN,NaN,sans_pz0,non_DEPHY
5526,fr.inra.agrosyst.api.entities.GrowingSystem_83...,70c570ed-0aa7-4db3-9b6e-6f57e789fca0,NaN,Agriculture conventionnelle,2025,NaN,NaN,sans_pz0,non_DEPHY
